# Spatial bbox queries

Fixed `N = 1_000_000` point cloud uniformly distributed in
`[0, 1000)³`.  Sweep bbox volume fraction `V ∈ {0.001, 0.01, 0.1, 1.0}`
of the domain and time each side reading just the points inside the
bbox:

- `zarr-vectors`: `open_zv(store)[0].filter(bbox=(lo, hi)).compute()`
  — only the chunks the bbox intersects are read.
- `pandas / CSV`: `pd.read_csv(path).query(...)` — no spatial index,
  must scan every row.

10 runs per `V`; each run draws a fresh random bbox location.  The
zarr-vectors line should stay roughly flat at small `V` and rise as
`V` approaches 1; the CSV line should be roughly flat at the
full-file-scan cost regardless of `V`.

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) — hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points
from zarr_vectors.lazy import open_zv

N = 1_000_000
V_FRACTIONS = [0.001, 0.01, 0.1, 1.0]
DOMAIN = 1000.0
CHUNK = (200.0, 200.0, 200.0)
BIN   = (50.0,  50.0,  50.0)
SEED  = 0


def _csv_path(prefix):
    return Path(tempfile.mkdtemp(prefix=f'csvbench_{prefix}_')) / 'points.csv'

## 2 · Write the shared input store and CSV baseline

In [ ]:
rng = np.random.default_rng(SEED)
positions = rng.uniform(0, DOMAIN, (N, 3)).astype(np.float32)

# zarr-vectors store: written once and reused across every (V, run).
zv_store = _new_store(f'bbox_zv_{N}')
write_points(zv_store, positions, chunk_shape=CHUNK, bin_shape=BIN)
zv_size_MB = _store_bytes(zv_store) / 1e6

# CSV baseline: same data dumped as text.
csv_path = _csv_path(f'bbox_csv_{N}')
pd.DataFrame({
    'x': positions[:, 0], 'y': positions[:, 1], 'z': positions[:, 2],
}).to_csv(csv_path, index=False)
csv_size_MB = csv_path.stat().st_size / 1e6

print(f'zarr-vectors store: {zv_size_MB:.2f} MB')
print(f'CSV baseline:       {csv_size_MB:.2f} MB')

## 3 · Run the sweep

In [ ]:
metrics = ['zv_s', 'csv_s', 'returned_count']
raw = {m: np.zeros((len(V_FRACTIONS), N_RUNS)) for m in metrics}

for i, v in enumerate(V_FRACTIONS):
    edge = DOMAIN * (v ** (1 / 3))
    for run in range(N_RUNS):
        lo = rng.uniform(0, DOMAIN - edge, 3).astype(np.float32)
        hi = (lo + edge).astype(np.float32)

        zv = open_zv(zv_store)
        t_zv, zv_out = _time(
            lambda: zv[0].filter(bbox=(lo, hi)).compute()
        )
        t_csv, csv_out = _time(
            lambda: pd.read_csv(csv_path).query(
                f'x >= {lo[0]} and x <= {hi[0]} and '
                f'y >= {lo[1]} and y <= {hi[1]} and '
                f'z >= {lo[2]} and z <= {hi[2]}'
            )
        )
        raw['zv_s'][i, run]            = t_zv
        raw['csv_s'][i, run]           = t_csv
        raw['returned_count'][i, run]  = zv_out['vertex_count']

rows = []
for i, v in enumerate(V_FRACTIONS):
    row = {'V': v, 'V_pct': v * 100}
    for m in metrics:
        mean, hw = _mean_ci95(raw[m][i])
        row[f'{m}_mean'] = round(mean, 4)
        row[f'{m}_hw']   = round(hw,   4)
    rows.append(row)
df = pd.DataFrame(rows)
shutil.rmtree(Path(zv_store).parent, ignore_errors=True)
shutil.rmtree(csv_path.parent, ignore_errors=True)

## 4 · Results

In [ ]:
df

## 5 · Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

SERIES = [
    ('zarr-vectors', 'zv',  'C0'),
    ('csv',          'csv', 'C1'),
]

# Panel 1: time vs V on log-log.
ax = axes[0]
for label, prefix, color in SERIES:
    mean = df[f'{prefix}_s_mean'].values
    hw   = df[f'{prefix}_s_hw'].values
    ax.fill_between(df['V'], mean - hw, mean + hw, color=color, alpha=0.2)
    ax.loglog(df['V'], mean, 'o-', color=color, label=label)
ax.set_xlabel('bbox volume fraction')
ax.set_ylabel('seconds')
ax.set_title(f'Bbox read time vs V (N = {N:,})')
ax.grid(True, which='both', alpha=0.3)
ax.legend()

# Panel 2: returned vertex count vs V (linearity check).
ax = axes[1]
mean = df['returned_count_mean'].values
ax.loglog(df['V'], mean, 'o-', color='C2', label='vertex_count')
ax.set_xlabel('bbox volume fraction')
ax.set_ylabel('returned vertices')
ax.set_title('Linearity check: count ∝ V')
ax.grid(True, which='both', alpha=0.3)

fig.suptitle(
    f'Spatial bbox queries — zarr-vectors vs CSV ({N_RUNS} runs, 95% CI)',
)
plt.tight_layout()